# Estimate behavioral-to-UPPT002 timing offsets

This notebook compares button press times from beh_data/s*/behDataMEG.mat with level changes in the matching participant's UPPT002 channel. The signed offset is UPPT002 detection time minus behavioral time: positive means the MEG channel changed later; negative means it changed earlier.

The behavioral time follows the repository's BehData_README and TFA_EMG notebook: (A/timing + T/cueDur) / 1000. It assumes both arrays are in milliseconds and T/cueDur has one value per trial. The MEG epochs are assumed to be in behavioral trial order.

UPPT002 is treated as a step/event channel. For each trial, the detector estimates baseline and noise robustly from the pre-cue interval, requires a sustained 20 ms level change away from baseline, rejects brief pulse-edge ringing, and ignores returns to baseline. The waveform QC distinguishes matched from unmatched detections and automatically includes trials with extra detections. Dynamic programming pairs expected and detected events in order, at most once each, and only within a configurable tolerance.

Assumptions to check before interpreting results:
- The existing TFA_EMG notebook shifts UPPT002 by 2.8 s, and the repository conversion creates EpochsArray with a 0 s start. The default below therefore places cue onset 2.8 s after each epoch starts. Confirm this against the saved epochs and trial waveform; change CUE_ONSET_AFTER_EPOCH_START_S if needed. This implementation does not use np.roll, which wraps the end of each trial to its beginning.
- The documentation describes an approximately 30 ms difference between MEG and PC timing but does not unambiguously specify its sign. The sign convention above makes the result directly inspectable.
- The repository does not document missing-value sentinels for A/timing; this code ignores non-finite entries only. It does not guess whether zero means a missing press.
- For a local export folder, set MEG_DATA_ROOT to its path; the loader accepts behavior under either root/s*/ or root/beh_data/s*/ and MEG epochs at root or root/s*/. Detector thresholds and the 250 ms match window should be reviewed with waveform and unmatched-event QC before using the estimate as a correction.


## 1. Configure the data location

The notebook uses the same participant and file naming conventions as the repository. Set MEG_DATA_ROOT to the folder that contains the subject data. It accepts either MEG_DATA_ROOT/beh_data/s05/ or MEG_DATA_ROOT/s05/ for the behavioral MAT files. The main Subject5_spm_mne-epo.fif file may be at the root or inside the subject folder; MNE follows its -1, -2, and later split parts.

The default root is the project network share. For a local copy, set the environment variable before launching Jupyter, or edit DATA_ROOT in the first code cell. The notebook does not copy or modify source data.

Two timing settings control the comparison:
- CUE_ONSET_AFTER_EPOCH_START_S maps the saved epoch clock to the cue-relative behavioral clock. The default 2.8 s follows the existing TFA_EMG workflow and should be confirmed for the dataset being analyzed.
- MATCH_TOLERANCE_S is the largest absolute difference allowed for a behavioral press and a detected UPPT002 event to be considered the same event. It is a matching limit, not a correction applied to the result.

In [ ]:
import os
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

# Same share and file naming convention used by the existing notebooks.
DATA_ROOT = Path(os.environ.get("MEG_DATA_ROOT", r"\\rdp.arc.ucl.ac.uk\ritd-ag-project-rd01pz-dbush99\Katja Button Press Data"))
BEHAVIOR_ROOT = DATA_ROOT / "beh_data"

# In the saved epochs, cue onset is assumed to be 2.8 s after epoch start.
CUE_ONSET_AFTER_EPOCH_START_S = 2.8
MATCH_TOLERANCE_S = 0.25

# Adaptive robust edge detector settings.
NOISE_SIGMA_MULTIPLIER = 8.0
RELATIVE_STEP_FRACTION = 0.02
PERSISTENCE_S = 0.020
REFRACTORY_S = 0.020

## 2. Load and interpret behavioral button times

The two files are MATLAB v7.3 files, read as HDF5. A/timing contains press times, while T/cueDur contains one cue-duration value per trial. The existing repository workflow defines each behavioral press time as:

behavior_time_s = (A/timing_ms + T/cueDur_ms) / 1000

The helper orients A/timing to (trials, press slots) because MATLAB/HDF5 axis order can differ from the layout expected by Python. It checks that one array axis equals the number of trials instead of assuming an orientation.

Non-finite values are treated as missing presses. The repository does not document whether zero is a missing-value code, so zero is retained as a timestamp. The computation also assumes behavioral trial order matches the order of epochs in the FIF file; the files do not provide a verified trial identifier join.

In [ ]:
def _as_trials_by_press(values, n_trials, name):
    """Orient a MATLAB array as (trials, press slots) without assuming HDF5 axis order."""
    values = np.asarray(values).squeeze()
    if values.ndim == 1 and values.size == n_trials:
        return values[:, None]
    if values.ndim != 2:
        raise ValueError(f"{name} should be a 2D trial-by-press array; got {values.shape}")
    if values.shape[0] == n_trials:
        return values
    if values.shape[1] == n_trials:
        return values.T
    raise ValueError(
        f"Could not identify the trial axis in {name} with shape {values.shape}; "
        f"expected one dimension to equal {n_trials}"
    )


def load_behavioral_press_times(subject_dir):
    """Return behavioral press times in seconds, shape (trials, press slots)."""
    with h5py.File(subject_dir / "behDataMEG.mat", "r") as f:
        timing_ms_raw = f["A/timing"][()]
    with h5py.File(subject_dir / "targetMEG.mat", "r") as f:
        cue_duration_ms = np.asarray(f["T/cueDur"][()]).squeeze().reshape(-1)

    timing_ms = _as_trials_by_press(timing_ms_raw, cue_duration_ms.size, "A/timing")
    if timing_ms.shape[0] != cue_duration_ms.size:
        raise ValueError("A/timing and T/cueDur have different trial counts")

    # T_press = A.timing(T, P) + T.cueDur(T), as described in BehData_README.txt.
    return (timing_ms + cue_duration_ms[:, None]) / 1000.0

## 3. Detect UPPT002 changes and match events

UPPT002 is treated as a pulse or step channel, not as a continuous neural signal. For each epoch the detector:

1. Estimates its baseline as the median of samples from 50 ms after epoch start to 50 ms before the assumed cue onset. The median absolute deviation (scaled by 1.4826) estimates robust baseline noise.
2. Sets a minimum step size to the largest of: eight times robust sample-to-sample noise, 2% of the 99.9th-percentile signal excursion, and a tiny numerical floor. This adapts the threshold to each epoch while reducing sensitivity to isolated outliers.
3. Checks candidate edges using 20 ms median windows before and after the edge. A level must remain away from baseline across this window. This suppresses brief ringing at pulse endings. Transitions returning to baseline are not counted as new presses; both positive and negative excursions can be detected.
4. Applies a 20 ms refractory interval to avoid counting multiple edges from one short transition.

The detector is run without using the behavioral timestamps, so the signal event list is independently obtained. The matching routine then performs an order-preserving, one-to-one assignment within MATCH_TOLERANCE_S: it first maximizes the number of matches, then minimizes the sum of absolute time differences. Unmatched behavioral entries and unmatched UPPT002 detections remain visible in QC instead of being forced into pairs.

The reported sign is:

offset_s = uppt002_time_s - behavior_time_s

Thus, a negative offset means UPPT002 changed before the behavioral log time; a positive offset means it changed later. The offset is measured relative to the assumed cue alignment.

In [ ]:
def _robust_sigma(values):
    values = np.asarray(values, dtype=float)
    center = np.median(values)
    return 1.4826 * np.median(np.abs(values - center))


def detect_uppt002_changes(signal, epoch_times, cue_onset_epoch_s):
    """Detect persistent UPPT002 level changes away from pre-cue baseline.\n\n    A 20 ms persistence window rejects brief ringing at pulse endings; only\n    the transition into a sustained non-baseline level is counted."""
    signal = np.asarray(signal, dtype=float).reshape(-1)
    epoch_times = np.asarray(epoch_times, dtype=float).reshape(-1)
    if signal.size != epoch_times.size:
        raise ValueError("UPPT002 samples and epoch times must have the same length")
    if signal.size < 10 or not np.isfinite(signal).all() or not np.isfinite(epoch_times).all():
        raise ValueError("UPPT002 and epoch times must be finite and contain enough samples")

    dt = float(np.median(np.diff(epoch_times)))
    if dt <= 0:
        raise ValueError("Epoch times must be strictly increasing")
    sfreq = 1.0 / dt

    # Use the stable portion before cue onset; leave a 50 ms guard before the cue.
    baseline_mask = (
        (epoch_times >= epoch_times[0] + 0.05)
        & (epoch_times < cue_onset_epoch_s - 0.05)
    )
    baseline_values = signal[baseline_mask]
    if baseline_values.size < max(10, int(round(0.05 * sfreq))):
        raise ValueError("Too few pre-cue samples; check cue-onset and epoch timing")

    baseline = float(np.median(baseline_values))
    baseline_noise = _robust_sigma(baseline_values)
    baseline_steps = np.diff(baseline_values)
    step_noise = _robust_sigma(baseline_steps) if baseline_steps.size else 0.0
    excursion = float(np.percentile(np.abs(signal - baseline), 99.9))
    numerical_floor = np.finfo(float).eps * max(1.0, float(np.max(np.abs(signal)))) * 10
    step_threshold = max(
        NOISE_SIGMA_MULTIPLIER * step_noise,
        RELATIVE_STEP_FRACTION * excursion,
        numerical_floor,
    )
    active_threshold = max(3.0 * baseline_noise, 0.25 * step_threshold, numerical_floor)

    persistence_n = max(3, int(np.ceil(PERSISTENCE_S * sfreq)))
    refractory_n = max(1, int(np.ceil(REFRACTORY_S * sfreq)))
    edge_sizes = np.abs(np.diff(signal, prepend=signal[0]))
    edge_indices = np.flatnonzero(edge_sizes >= step_threshold)

    event_samples = []
    last_event_sample = -refractory_n
    for sample in edge_indices:
        if sample < persistence_n or sample + persistence_n > signal.size:
            continue
        if sample - last_event_sample < refractory_n:
            continue

        before = float(np.median(signal[sample - persistence_n:sample]))
        after = float(np.median(signal[sample:sample + persistence_n]))
        if abs(after - before) < step_threshold:
            continue
        # A reset is a change whose post-edge level is back at baseline.
        if abs(after - baseline) <= active_threshold:
            continue

        event_samples.append(int(sample))
        last_event_sample = int(sample)

    event_samples = np.asarray(event_samples, dtype=int)
    event_times = epoch_times[event_samples] - cue_onset_epoch_s
    diagnostics = {
        "baseline": baseline,
        "baseline_noise_sigma": baseline_noise,
        "step_threshold": step_threshold,
        "active_threshold": active_threshold,
        "n_detected": len(event_samples),
    }
    return event_times, event_samples, diagnostics


def match_events_in_order(behavior_times, detected_times, tolerance_s):
    """Maximum-cardinality, minimum-offset, order-preserving one-to-one match."""
    behavior_times = np.asarray(behavior_times, dtype=float)
    detected_times = np.asarray(detected_times, dtype=float)
    behavior_order = np.flatnonzero(np.isfinite(behavior_times))
    behavior_order = behavior_order[
        np.argsort(behavior_times[behavior_order], kind="stable")
    ]
    detection_order = np.argsort(detected_times, kind="stable")
    behavior_sorted = behavior_times[behavior_order]
    detection_sorted = detected_times[detection_order]

    n, m = behavior_sorted.size, detection_sorted.size
    n_matches = np.zeros((n + 1, m + 1), dtype=int)
    total_cost = np.zeros((n + 1, m + 1), dtype=float)
    action = np.zeros((n + 1, m + 1), dtype=np.int8)
    action[1:, 0] = 1
    action[0, 1:] = 2

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            options = [
                (n_matches[i - 1, j], total_cost[i - 1, j], 1),
                (n_matches[i, j - 1], total_cost[i, j - 1], 2),
            ]
            difference = abs(behavior_sorted[i - 1] - detection_sorted[j - 1])
            if difference <= tolerance_s:
                options.append(
                    (n_matches[i - 1, j - 1] + 1,
                     total_cost[i - 1, j - 1] + difference, 3)
                )
            best = max(options, key=lambda item: (item[0], -item[1], item[2] == 3))
            n_matches[i, j], total_cost[i, j], action[i, j] = best

    pairs = []
    i, j = n, m
    while i > 0 or j > 0:
        step = action[i, j]
        if step == 3:
            pairs.append((int(behavior_order[i - 1]), int(detection_order[j - 1])))
            i -= 1
            j -= 1
        elif step == 1:
            i -= 1
        elif step == 2:
            j -= 1
        else:
            break
    return list(reversed(pairs))

## 4. Analyze one participant and summarize offsets

For each participant folder, the analysis checks that a matching main FIF file and the UPPT002 channel exist, and that the epoch count equals the behavioral trial count. It stops on a count mismatch because pairing by row would then be unsafe.

The FIF is opened with preload=False, and only UPPT002 is read for analysis. This avoids loading every MEG channel from large, split epoch files into memory.

The event table has one row per matched press and retains its trial, behavioral time, UPPT002 time, and signed offset. The trial QC table reports expected presses, detected transitions, matches, and unmatched counts, plus the detector thresholds for review.

The primary participant mean is calculated by first averaging matched offsets within each trial and then averaging those trial means. That gives every trial equal weight even when trials have different numbers of presses. The notebook also reports event-weighted summaries for comparison. The displayed 95% interval is an approximate normal interval based on trial means; interpret it as descriptive QC, especially if trials are dependent or many presses are unmatched.

In [ ]:
def _find_epochs_file(data_root, subject_id):
    """Find the main FIF file; MNE follows its -1/-2 split files automatically."""
    subject_number = int(subject_id[1:])
    search_dirs = [data_root / subject_id, data_root]
    candidates = []
    for folder in search_dirs:
        if not folder.is_dir():
            continue
        for path in folder.glob("Subject*_spm_mne-epo.fif"):
            number_text = path.name[len("Subject"):].split("_", 1)[0]
            if number_text.isdigit() and int(number_text) == subject_number:
                candidates.append(path)
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Expected one main FIF file for {subject_id} under {data_root}, "
            f"found {len(candidates)}: {candidates}"
        )
    return candidates[0]


def analyze_subject(subject_dir):
    """Analyze one beh_data/s* folder and return event and trial QC records."""
    subject_id = subject_dir.name
    behavior_times = load_behavioral_press_times(subject_dir)

    epochs_path = _find_epochs_file(DATA_ROOT, subject_id)
    epochs = mne.read_epochs(epochs_path, preload=False, verbose="ERROR")
    if "UPPT002" not in epochs.ch_names:
        raise ValueError(f"{epochs_path} has no UPPT002 channel")
    if len(epochs) != behavior_times.shape[0]:
        raise ValueError(
            f"{subject_id}: {len(epochs)} MEG epochs but "
            f"{behavior_times.shape[0]} behavioral trials; trial matching is unsafe"
        )

    epoch_times = epochs.times
    cue_onset_epoch_s = epoch_times[0] + CUE_ONSET_AFTER_EPOCH_START_S
    channel = epochs.get_data(picks=["UPPT002"])[:, 0, :]
    match_records, trial_records, detected_by_trial = [], [], []

    for trial_idx, signal in enumerate(channel):
        detected_times, _, diag = detect_uppt002_changes(
            signal, epoch_times, cue_onset_epoch_s
        )
        detected_by_trial.append(detected_times)
        expected = behavior_times[trial_idx]
        pairs = match_events_in_order(expected, detected_times, MATCH_TOLERANCE_S)

        for behavior_slot, detected_idx in pairs:
            behavior_time = float(expected[behavior_slot])
            uppt_time = float(detected_times[detected_idx])
            match_records.append(
                {
                    "participant": subject_id,
                    "trial": trial_idx + 1,
                    "press_slot": behavior_slot + 1,
                    "behavior_time_s": behavior_time,
                    "uppt002_time_s": uppt_time,
                    # Sign convention: positive means the UPPT002 transition came later.
                    "offset_s": uppt_time - behavior_time,
                }
            )

        n_expected = int(np.isfinite(expected).sum())
        n_matched = len(pairs)
        trial_records.append(
            {
                "participant": subject_id,
                "trial": trial_idx + 1,
                "n_behavior_expected": n_expected,
                "n_uppt_detected": len(detected_times),
                "n_matched": n_matched,
                "n_behavior_unmatched": n_expected - n_matched,
                "n_uppt_unmatched": len(detected_times) - n_matched,
                "baseline_noise_sigma": diag["baseline_noise_sigma"],
                "step_threshold": diag["step_threshold"],
            }
        )

    return {
        "participant": subject_id,
        "epochs_path": epochs_path,
        "epoch_times": epoch_times.copy(),
        "cue_onset_epoch_s": cue_onset_epoch_s,
        "behavior_times": behavior_times,
        "detected_by_trial": detected_by_trial,
        "matches": match_records,
        "trial_qc": trial_records,
    }


def summarize_participants(results):
    """Summarize offsets, giving each trial equal weight in the primary mean."""
    match_columns = [
        "participant", "trial", "press_slot", "behavior_time_s", "uppt002_time_s", "offset_s"
    ]
    qc_columns = [
        "participant", "trial", "n_behavior_expected", "n_uppt_detected", "n_matched",
        "n_behavior_unmatched", "n_uppt_unmatched", "baseline_noise_sigma", "step_threshold"
    ]
    matches = pd.DataFrame(
        [row for result in results for row in result["matches"]], columns=match_columns
    )
    trial_qc = pd.DataFrame(
        [row for result in results for row in result["trial_qc"]], columns=qc_columns
    )
    summaries = []

    for result in results:
        participant = result["participant"]
        participant_trials = trial_qc[trial_qc["participant"] == participant]
        participant_matches = matches[matches["participant"] == participant]
        if participant_matches.empty:
            trial_means = np.array([], dtype=float)
            event_offsets = np.array([], dtype=float)
        else:
            trial_means = (
                participant_matches.groupby("trial", sort=True)["offset_s"].mean().to_numpy()
            )
            event_offsets = participant_matches["offset_s"].to_numpy()

        n_trials = trial_means.size
        trial_mean = float(np.mean(trial_means)) if n_trials else np.nan
        trial_sd = float(np.std(trial_means, ddof=1)) if n_trials > 1 else np.nan
        trial_sem = trial_sd / np.sqrt(n_trials) if n_trials > 1 else np.nan
        expected_count = int(participant_trials["n_behavior_expected"].sum())
        matched_count = int(participant_trials["n_matched"].sum())
        summaries.append(
            {
                "participant": participant,
                "n_trials": int(len(participant_trials)),
                "n_trials_with_match": int(n_trials),
                "n_expected_events": expected_count,
                "n_matched_events": matched_count,
                "match_rate": matched_count / expected_count if expected_count else np.nan,
                # Primary result: mean of within-trial means, so each trial has equal weight.
                "mean_trial_offset_s": trial_mean,
                "median_trial_offset_s": float(np.median(trial_means)) if n_trials else np.nan,
                "sd_trial_offset_s": trial_sd,
                "approx_95ci_low_s": trial_mean - 1.96 * trial_sem if n_trials > 1 else np.nan,
                "approx_95ci_high_s": trial_mean + 1.96 * trial_sem if n_trials > 1 else np.nan,
                # Event-weighted values are useful when trials have different match counts.
                "mean_event_offset_s": float(np.mean(event_offsets)) if event_offsets.size else np.nan,
                "median_event_offset_s": float(np.median(event_offsets)) if event_offsets.size else np.nan,
                "mad_event_offset_s": (
                    float(np.median(np.abs(event_offsets - np.median(event_offsets))))
                    if event_offsets.size else np.nan
                ),
                "n_unmatched_behavior": int(participant_trials["n_behavior_unmatched"].sum()),
                "n_unmatched_uppt": int(participant_trials["n_uppt_unmatched"].sum()),
            }
        )

    return pd.DataFrame(summaries), matches, trial_qc

## 5. Run all available participants

This cell finds folders named s followed by digits and containing both behavioral MAT files. It checks the beh_data layout first, then the local-export layout with subject folders directly under the data root. Each discovered participant is analyzed using the matching FIF file.

The summary table gives the participant-level offset estimates and match coverage. The following QC counts are summed by participant:
- n_behavior_unmatched: behavioral presses without a detected event inside the matching window.
- n_uppt_unmatched: detected UPPT002 transitions not assigned to a behavioral press.

A high match rate does not by itself prove that the cue alignment or event interpretation is correct; inspect the waveform plots and unmatched counts as well.

In [ ]:
# Analyze every available beh_data/s* folder. The matching MEG epoch file is
# expected in the same participant folder one level above beh_data.
subject_dirs = []
for behavior_root in (BEHAVIOR_ROOT, DATA_ROOT):
    subject_dirs = sorted(
        path for path in behavior_root.glob("s*")
        if path.is_dir()
        and path.name[1:].isdigit()
        and (path / "behDataMEG.mat").is_file()
        and (path / "targetMEG.mat").is_file()
    )
    if subject_dirs:
        break
if not subject_dirs:
    raise FileNotFoundError(
        f"No s* folder with behDataMEG.mat and targetMEG.mat under "
        f"{BEHAVIOR_ROOT} or {DATA_ROOT}"
    )

results = [analyze_subject(subject_dir) for subject_dir in subject_dirs]
summary_table, event_table, trial_qc_table = summarize_participants(results)

display(summary_table.round(4))
display(
    trial_qc_table.groupby("participant")[
        ["n_behavior_unmatched", "n_uppt_unmatched"]
    ].sum()
)

## 6. Read the QC figures

The cohort figure shows each participant's trial-weighted mean with its approximate 95% interval, then a histogram of all matched event offsets. A cluster away from zero suggests a consistent timing difference under the chosen cue alignment. The histogram is event-weighted, so participants or trials with more matched presses contribute more points.

The waveform figure shows signal time relative to the assumed cue:
- Red dashed lines are behavioral button-press times.
- Green triangles are matched UPPT002 detections.
- Orange crosses are unmatched UPPT002 detections.

By default, it includes a few representative matched trials and up to two trials with extra detections. Set QC_SUBJECT to a participant folder such as s05; pass trial_numbers to plot_subject_waveforms to inspect particular trials. Check pulse shape, timing alignment, and all unmatched events before treating the mean as a hardware correction.

In [ ]:
# Cohort QC: trial-weighted means with approximate 95% CIs and all matched
# event offsets. A negative offset means UPPT002 changed before the PC log time.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_summary = summary_table.dropna(subset=["mean_trial_offset_s"]).reset_index(drop=True)
x = np.arange(len(plot_summary))
lower = plot_summary["mean_trial_offset_s"] - plot_summary["approx_95ci_low_s"]
upper = plot_summary["approx_95ci_high_s"] - plot_summary["mean_trial_offset_s"]
axes[0].errorbar(
    x, plot_summary["mean_trial_offset_s"], yerr=np.vstack([lower, upper]),
    fmt="o", capsize=3,
)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_xticks(x, plot_summary["participant"], rotation=45)
axes[0].set_ylabel("UPPT002 − behavioral time (s)")
axes[0].set_title("Participant mean of within-trial offsets")
axes[0].grid(alpha=0.25)

if not event_table.empty:
    axes[1].hist(event_table["offset_s"], bins=40, color="slateblue", alpha=0.8)
    axes[1].axvline(0, color="black", linewidth=1)
    axes[1].axvline(event_table["offset_s"].mean(), color="darkorange", linestyle="--")
axes[1].set_xlabel("UPPT002 − behavioral time (s)")
axes[1].set_ylabel("Matched presses")
axes[1].set_title("All matched event offsets")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()


def plot_subject_waveforms(result, trial_numbers=None):
    """Overlay behavioral timestamps and detected UPPT002 changes for selected trials."""
    epochs = mne.read_epochs(result["epochs_path"], preload=False, verbose="ERROR")
    channel = epochs.get_data(picks=["UPPT002"])[:, 0, :]
    relative_times = epochs.times - result["cue_onset_epoch_s"]

    if trial_numbers is None:
        matched_trials = sorted({row["trial"] for row in result["matches"]})
        if not matched_trials:
            print(f"{result['participant']}: no matched trials to plot")
            return
        trial_means = pd.DataFrame(result["matches"]).groupby("trial")["offset_s"].mean()
        median_trial = int((trial_means - trial_means.median()).abs().idxmin())
        largest_trial = int(trial_means.abs().idxmax())
        trial_qc = pd.DataFrame(result["trial_qc"])
        unmatched_trials = trial_qc.loc[
            trial_qc["n_uppt_unmatched"] > 0, "trial"
        ].astype(int).tolist()
        trial_numbers = list(dict.fromkeys(
            [matched_trials[0], median_trial, largest_trial, *unmatched_trials[:2]]
        ))

    trial_numbers = [number for number in trial_numbers if 1 <= number <= len(channel)]
    fig, axes = plt.subplots(len(trial_numbers), 1, figsize=(13, 3.2 * len(trial_numbers)))
    axes = np.atleast_1d(axes)
    for ax, trial_number in zip(axes, trial_numbers):
        trial_idx = trial_number - 1
        ax.plot(relative_times, channel[trial_idx], color="black", linewidth=0.8)
        for slot, behavior_time in enumerate(result["behavior_times"][trial_idx]):
            if np.isfinite(behavior_time):
                ax.axvline(
                    behavior_time, color="tab:red", linestyle="--", alpha=0.7,
                    label="Behavioral time" if slot == 0 else None,
                )
        detected = result["detected_by_trial"][trial_idx]
        matched_times = np.asarray([
            row["uppt002_time_s"] for row in result["matches"]
            if row["trial"] == trial_number
        ])
        matched_mask = np.asarray([
            np.any(np.isclose(matched_times, event_time, atol=1e-9))
            for event_time in detected
        ], dtype=bool)
        if detected.size:
            values = np.interp(detected, relative_times, channel[trial_idx])
            ax.scatter(
                detected[matched_mask], values[matched_mask], marker="v",
                color="tab:green", zorder=3, label="Matched detection",
            )
            ax.scatter(
                detected[~matched_mask], values[~matched_mask], marker="x",
                color="tab:orange", zorder=3, label="Unmatched detection",
            )
        n_unmatched = int((~matched_mask).sum())
        ax.set_title(f"{result['participant']} trial {trial_number} ({n_unmatched} unmatched detections)")
        ax.set_xlabel("Time relative to cue onset (s)")
        ax.set_ylabel("UPPT002 (native units)")
        ax.grid(alpha=0.25)
        ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


# Set this to a folder name such as s09 to choose a participant; None selects the first.
QC_SUBJECT = None
qc_result = next(
    result for result in results
    if QC_SUBJECT is None or result["participant"] == QC_SUBJECT
)
plot_subject_waveforms(qc_result)